# Generating Combinatorial Sub-Components from Connected Topologies

This notebook demonstrates how to generate multiple sub-components from a connected topology
(e.g., `CellComplex`, `Cell`, `Shell`, or `Graph`).

**Adapted from topologicpy CombinatorialSubComponents example.**

In topologicpy, `Topology.SubCombinations()` generates sub-topologies of varying sizes.
Since topologic_fast may not have this exact method, we implement similar functionality
using Python's `itertools.combinations`.

This tutorial shows how to:
1. Create a CellComplex (3x3x3 grid of cells)
2. Generate combinatorial subsets of cells
3. Create sub-CellComplexes from selected cells
4. Visualize the results using Plotly

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
from itertools import combinations
import random

print("Libraries imported successfully.")

## 1. Create a CellComplex (3x3x3 Grid)

We create a CellComplex by building a 3x3x3 grid of box cells and combining them.

In [ ]:
# Create a 3x3x3 grid of cells
def create_prism_cellcomplex(u_sides=3, v_sides=3, w_sides=3, size=1.0):
    """
    Create a CellComplex resembling a 3D grid.
    
    Parameters:
        u_sides: Number of cells in X direction
        v_sides: Number of cells in Y direction
        w_sides: Number of cells in Z direction
        size: Size of each cell
    """
    cells = []
    
    for i in range(u_sides):
        for j in range(v_sides):
            for k in range(w_sides):
                # Create a box cell at position (i, j, k)
                x = i * size
                y = j * size
                z = k * size
                
                cell = tf.Cell.Box(x, y, z, size, size, size)
                cells.append(cell)
    
    # Combine all cells into a CellComplex
    cellcomplex = tf.CellComplex.ByCells(cells)
    
    return cellcomplex, cells

# Create the 3x3x3 CellComplex
cc, individual_cells = create_prism_cellcomplex(u_sides=3, v_sides=3, w_sides=3)

print(f"Created CellComplex:")
print(f"  Total cells: {cc.NumCells()}")
print(f"  Total faces: {cc.NumFaces()}")
print(f"  Total volume: {cc.Volume():.2f}")

## 2. Visualize the Base CellComplex

In [ ]:
def visualize_cellcomplex_3d(cellcomplex, title="CellComplex", opacity=0.3, show_edges=True):
    """Create a 3D visualization of a CellComplex."""
    fig = go.Figure()
    
    cells = cellcomplex.Cells()
    
    for i, cell in enumerate(cells):
        faces = cell.Faces()
        
        for face in faces:
            vertices = face.Vertices()
            coords = [v.Coordinates() for v in vertices]
            
            if len(coords) >= 3:
                x = [c[0] for c in coords]
                y = [c[1] for c in coords]
                z = [c[2] for c in coords]
                
                fig.add_trace(go.Mesh3d(
                    x=x, y=y, z=z,
                    color='lightblue',
                    opacity=opacity,
                    alphahull=0,
                    showlegend=False,
                    hoverinfo='skip'
                ))
                
                if show_edges:
                    # Draw edges
                    for k in range(len(coords)):
                        p1 = coords[k]
                        p2 = coords[(k + 1) % len(coords)]
                        fig.add_trace(go.Scatter3d(
                            x=[p1[0], p2[0]],
                            y=[p1[1], p2[1]],
                            z=[p1[2], p2[2]],
                            mode='lines',
                            line=dict(color='darkblue', width=2),
                            showlegend=False,
                            hoverinfo='skip'
                        ))
    
    fig.update_layout(
        title=title,
        scene=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0))
        ),
        width=700,
        height=600
    )
    
    return fig

fig_base = visualize_cellcomplex_3d(cc, "Base CellComplex (3x3x3 = 27 cells)")
fig_base.show()

## 3. Generate Sub-Combinations of Cells

In topologicpy, `Topology.SubCombinations()` generates subsets of cells/faces.
Since topologic_fast may not have this method, we implement it using Python.

We will generate combinations of cells with sizes ranging from `minSize` to `maxSize`.

In [ ]:
# NOTE: In topologicpy, you would use:
#   cc_combinations = Topology.SubCombinations(
#       cc,
#       minSize=2,
#       maxSize=6,
#       maxCombinations=10,
#       timeLimit=10
#   )
#
# Since this may not be available in topologic_fast, we implement it:

def generate_sub_combinations(cells, min_size=2, max_size=6, max_combinations=10, seed=42):
    """
    Generate sub-combinations of cells.
    
    Parameters:
        cells: List of cells from a CellComplex
        min_size: Minimum number of cells in a combination
        max_size: Maximum number of cells in a combination
        max_combinations: Maximum total combinations to return
        seed: Random seed for reproducibility
    
    Returns:
        List of cell combinations (each combination is a list of cells)
    """
    random.seed(seed)
    n_cells = len(cells)
    
    # Clamp sizes to valid range
    min_size = max(1, min(min_size, n_cells))
    max_size = max(min_size, min(max_size, n_cells))
    
    all_combinations = []
    
    # Generate combinations for each size
    for size in range(min_size, max_size + 1):
        # Get all combinations of this size
        combos = list(combinations(range(n_cells), size))
        
        # Convert to cell lists
        for combo in combos:
            cell_combo = [cells[i] for i in combo]
            all_combinations.append(cell_combo)
    
    # Shuffle and limit
    random.shuffle(all_combinations)
    
    # Try to balance across sizes
    balanced = []
    by_size = {}
    for combo in all_combinations:
        s = len(combo)
        if s not in by_size:
            by_size[s] = []
        by_size[s].append(combo)
    
    # Round-robin selection from each size
    sizes = sorted(by_size.keys())
    idx = {s: 0 for s in sizes}
    
    while len(balanced) < max_combinations:
        added = False
        for s in sizes:
            if idx[s] < len(by_size[s]) and len(balanced) < max_combinations:
                balanced.append(by_size[s][idx[s]])
                idx[s] += 1
                added = True
        if not added:
            break
    
    return balanced

# Get cells from CellComplex
cc_cells = cc.Cells()

# Generate sub-combinations
cell_combinations = generate_sub_combinations(
    cc_cells,
    min_size=2,
    max_size=6,
    max_combinations=10
)

print(f"Generated {len(cell_combinations)} cell combinations:")
print("-" * 40)
for i, combo in enumerate(cell_combinations):
    print(f"  Combination {i+1}: {len(combo)} cells")

## 4. Create Sub-CellComplexes from Combinations

We convert each combination of cells into a CellComplex.

In [ ]:
def create_cellcomplex_from_cells(cells):
    """Create a CellComplex from a list of cells."""
    if len(cells) == 0:
        return None
    if len(cells) == 1:
        # Single cell - return as is or wrap
        return cells[0]
    
    try:
        return tf.CellComplex.ByCells(cells)
    except:
        # If ByCells fails, return list of cells
        return cells

# Create sub-CellComplexes
cc_combinations = []
for combo in cell_combinations:
    sub_cc = create_cellcomplex_from_cells(combo)
    cc_combinations.append(sub_cc)

print(f"Created {len(cc_combinations)} sub-CellComplexes")

## 5. Visualize a Sub-Combination

We visualize one of the generated combinations overlaid on the original CellComplex wireframe.

In [ ]:
def visualize_combination(base_cc, combo_cells, combo_index, combo_color='beige'):
    """Visualize a combination of cells against the base CellComplex."""
    fig = go.Figure()
    
    # Draw base CellComplex edges (wireframe)
    base_cells = base_cc.Cells()
    drawn_edges = set()
    
    for cell in base_cells:
        edges = cell.Edges()
        for edge in edges:
            vertices = edge.Vertices()
            if len(vertices) == 2:
                p1 = vertices[0].Coordinates()
                p2 = vertices[1].Coordinates()
                
                # Create edge key for deduplication
                edge_key = tuple(sorted([tuple(p1), tuple(p2)]))
                if edge_key not in drawn_edges:
                    drawn_edges.add(edge_key)
                    fig.add_trace(go.Scatter3d(
                        x=[p1[0], p2[0]],
                        y=[p1[1], p2[1]],
                        z=[p1[2], p2[2]],
                        mode='lines',
                        line=dict(color='lightgray', width=1),
                        showlegend=False,
                        hoverinfo='skip'
                    ))
    
    # Draw combination cells
    for cell in combo_cells:
        faces = cell.Faces()
        
        for face in faces:
            vertices = face.Vertices()
            coords = [v.Coordinates() for v in vertices]
            
            if len(coords) >= 3:
                x = [c[0] for c in coords]
                y = [c[1] for c in coords]
                z = [c[2] for c in coords]
                
                fig.add_trace(go.Mesh3d(
                    x=x, y=y, z=z,
                    color=combo_color,
                    opacity=0.9,
                    alphahull=0,
                    showlegend=False,
                    hoverinfo='skip'
                ))
    
    fig.update_layout(
        title=f"Combination {combo_index + 1}: {len(combo_cells)} cells",
        scene=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0))
        ),
        width=700,
        height=600
    )
    
    return fig

# Choose a combination to visualize (change n to see different ones)
n = 4
n = max(0, min(n, len(cell_combinations) - 1))

fig_combo = visualize_combination(cc, cell_combinations[n], n)
fig_combo.show()

## 6. Face-Based Sub-Combinations (Creating Shells)

Instead of selecting cells, we can select faces to create shells.
This creates more varied geometric subsets.

In [ ]:
# NOTE: In topologicpy, you would use:
#   shell_combinations = Topology.SubCombinations(
#       cc,
#       minSize=2,
#       maxSize=6,
#       maxCombinations=10,
#       subTopologyType="face",
#       timeLimit=30
#   )
#
# We implement this for topologic_fast:

# Get all faces from the CellComplex
cc_faces = cc.Faces()
print(f"CellComplex has {len(cc_faces)} faces")

# Generate face combinations
face_combinations = generate_sub_combinations(
    cc_faces,
    min_size=2,
    max_size=6,
    max_combinations=10,
    seed=123
)

print(f"\nGenerated {len(face_combinations)} face combinations:")
for i, combo in enumerate(face_combinations):
    print(f"  Face combination {i+1}: {len(combo)} faces")

In [ ]:
def create_shell_from_faces(faces):
    """Create a Shell from a list of faces."""
    if len(faces) == 0:
        return None
    
    try:
        return tf.Shell.ByFaces(faces)
    except:
        # If ByFaces fails, return the faces as-is
        return faces

# Create shells from face combinations
shell_combinations = [create_shell_from_faces(combo) for combo in face_combinations]
print(f"Created {len(shell_combinations)} shell combinations")

In [ ]:
def visualize_face_combination(base_cc, combo_faces, combo_index, combo_color='orange'):
    """Visualize a combination of faces against the base CellComplex."""
    fig = go.Figure()
    
    # Draw base CellComplex edges (wireframe)
    base_cells = base_cc.Cells()
    drawn_edges = set()
    
    for cell in base_cells:
        edges = cell.Edges()
        for edge in edges:
            vertices = edge.Vertices()
            if len(vertices) == 2:
                p1 = vertices[0].Coordinates()
                p2 = vertices[1].Coordinates()
                
                edge_key = tuple(sorted([tuple(p1), tuple(p2)]))
                if edge_key not in drawn_edges:
                    drawn_edges.add(edge_key)
                    fig.add_trace(go.Scatter3d(
                        x=[p1[0], p2[0]],
                        y=[p1[1], p2[1]],
                        z=[p1[2], p2[2]],
                        mode='lines',
                        line=dict(color='lightgray', width=1),
                        showlegend=False,
                        hoverinfo='skip'
                    ))
    
    # Draw combination faces
    for face in combo_faces:
        vertices = face.Vertices()
        coords = [v.Coordinates() for v in vertices]
        
        if len(coords) >= 3:
            x = [c[0] for c in coords]
            y = [c[1] for c in coords]
            z = [c[2] for c in coords]
            
            fig.add_trace(go.Mesh3d(
                x=x, y=y, z=z,
                color=combo_color,
                opacity=0.9,
                alphahull=0,
                showlegend=False,
                hoverinfo='skip'
            ))
            
            # Draw face edges
            for k in range(len(coords)):
                p1 = coords[k]
                p2 = coords[(k + 1) % len(coords)]
                fig.add_trace(go.Scatter3d(
                    x=[p1[0], p2[0]],
                    y=[p1[1], p2[1]],
                    z=[p1[2], p2[2]],
                    mode='lines',
                    line=dict(color='darkorange', width=3),
                    showlegend=False,
                    hoverinfo='skip'
                ))
    
    fig.update_layout(
        title=f"Face Combination {combo_index + 1}: {len(combo_faces)} faces",
        scene=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0))
        ),
        width=700,
        height=600
    )
    
    return fig

# Visualize a face combination
n = 0
n = max(0, min(n, len(face_combinations) - 1))

fig_faces = visualize_face_combination(cc, face_combinations[n], n)
fig_faces.show()

## 7. Graph Sub-Combinations

We can also create sub-graphs from a topology's dual graph.
The graph represents connectivity between cells.

In [ ]:
# Create a graph from the CellComplex
graph = tf.Graph.ByTopology(cc)

print(f"Graph from CellComplex:")
print(f"  Vertices (cells): {graph.Order()}")
print(f"  Edges (connections): {graph.Size()}")
print(f"  Density: {graph.Density():.3f}")

In [ ]:
# Get graph vertices
graph_vertices = graph.Vertices()

# Generate sub-graph combinations (subsets of vertices)
# NOTE: In topologicpy, you would use:
#   graph_combinations = Topology.SubCombinations(
#       graph,
#       minSize=2,
#       maxSize=8,
#       maxCombinations=50,
#       timeLimit=60
#   )

vertex_combinations = generate_sub_combinations(
    graph_vertices,
    min_size=2,
    max_size=8,
    max_combinations=20,
    seed=456
)

print(f"Generated {len(vertex_combinations)} sub-graph vertex combinations")

In [ ]:
def visualize_graph_combination(graph, combo_vertices, combo_index):
    """Visualize a sub-graph combination."""
    fig = go.Figure()
    
    # Get all graph vertices and edges
    all_vertices = graph.Vertices()
    all_edges = graph.Edges()
    
    # Draw all edges (light)
    for edge in all_edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='lightgray', width=2),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw all vertices (light)
    all_coords = [v.Coordinates() for v in all_vertices]
    fig.add_trace(go.Scatter3d(
        x=[c[0] for c in all_coords],
        y=[c[1] for c in all_coords],
        z=[c[2] for c in all_coords],
        mode='markers',
        marker=dict(size=4, color='lightgray'),
        showlegend=False,
        hoverinfo='skip'
    ))
    
    # Get coordinates of combo vertices
    combo_coords = [v.Coordinates() for v in combo_vertices]
    combo_set = set(tuple(c) for c in combo_coords)
    
    # Draw edges between combo vertices (highlighted)
    for edge in all_edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            
            if tuple(p1) in combo_set and tuple(p2) in combo_set:
                fig.add_trace(go.Scatter3d(
                    x=[p1[0], p2[0]],
                    y=[p1[1], p2[1]],
                    z=[p1[2], p2[2]],
                    mode='lines',
                    line=dict(color='black', width=5),
                    showlegend=False,
                    hoverinfo='skip'
                ))
    
    # Draw combo vertices (highlighted)
    fig.add_trace(go.Scatter3d(
        x=[c[0] for c in combo_coords],
        y=[c[1] for c in combo_coords],
        z=[c[2] for c in combo_coords],
        mode='markers',
        marker=dict(size=10, color='blue', line=dict(color='darkblue', width=2)),
        name=f'Selected ({len(combo_vertices)} vertices)',
        hoverinfo='text',
        hovertext=[f"Vertex {i}" for i in range(len(combo_vertices))]
    ))
    
    fig.update_layout(
        title=f"Sub-Graph Combination {combo_index + 1}: {len(combo_vertices)} vertices",
        scene=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0))
        ),
        width=700,
        height=600
    )
    
    return fig

# Visualize a sub-graph combination
n = 10
n = max(0, min(n, len(vertex_combinations) - 1))

fig_graph = visualize_graph_combination(graph, vertex_combinations[n], n)
fig_graph.show()

## Summary

This notebook demonstrated how to generate combinatorial sub-components from connected topologies using topologic_fast.

### Key Operations:

1. **CellComplex Creation**: Built a 3x3x3 grid of cells using `tf.Cell.Box()` and `tf.CellComplex.ByCells()`
2. **Cell Sub-Combinations**: Generated random subsets of cells using Python's `itertools.combinations`
3. **Face Sub-Combinations**: Selected faces to create shell-like sub-topologies
4. **Graph Sub-Combinations**: Created sub-graphs from the topology's dual graph

### Differences from topologicpy:

- **No `Topology.SubCombinations()`**: Implemented using Python's `itertools.combinations`
- **No `CellComplex.Prism()`**: Built manually using loops and `Cell.Box()`
- **No `Plotly.DataByTopology()`**: Created Plotly visualizations manually

### Applications:

- Exploring design alternatives
- Generating training data for machine learning
- Understanding topological relationships
- Testing connectivity patterns